In [82]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torchvision import datasets
from torchvision.transforms import transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
from torchsummary import summary
from torch.utils.tensorboard import SummaryWriter

In [65]:
data_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Resize(32),
        transforms.Normalize((0.5,),(1.0,))             # 간이 정규화 (원랜 이렇게 하면 좋은건 아님)
    ]
)

train_data = datasets.MNIST(root="./data", train=True, download=True, transform=data_transform)
test_data = datasets.MNIST(root="./data", train=False, download=True, transform=data_transform)

train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Resize(size=32, interpolation=bilinear, max_size=None, antialias=True)
               Normalize(mean=(0.5,), std=(1.0,))
           )

In [66]:
train_data.data.shape

torch.Size([60000, 28, 28])

In [67]:
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

In [68]:
next(iter(train_loader))[0].shape

torch.Size([128, 1, 32, 32])

In [71]:
class LeNet5(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, stride=1)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1)
        self.conv3 = nn.Conv2d(in_channels=16, out_channels=120, kernel_size=5, stride=1)
        self.fc1 = nn.Linear(120, 84)
        self.fc2 = nn.Linear(84, 10)
    
    def forward(self, x):
        x = F.tanh(self.conv1(x))
        x = F.max_pool2d(x, 2, 2)
        x = F.tanh(self.conv2(x))
        x = F.max_pool2d(x, 2, 2)
        x = F.tanh(self.conv3(x))
        x = x.view(-1,120)              # flatten 형태 잘 맞추기가 가장 중요
        x = F.tanh(self.fc1(x))
        x = F.tanh(self.fc2(x))
        
        return x



model = LeNet5()
model

LeNet5(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (conv3): Conv2d(16, 120, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=120, out_features=84, bias=True)
  (fc2): Linear(in_features=84, out_features=10, bias=True)
)

In [84]:
num_epochs = 10
lr = 1e-4

In [77]:
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=lr)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

LeNet5(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (conv3): Conv2d(16, 120, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=120, out_features=84, bias=True)
  (fc2): Linear(in_features=84, out_features=10, bias=True)
)

In [80]:
summary(model, input_size=(1, 64, 64))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1            [-1, 6, 60, 60]             156
            Conv2d-2           [-1, 16, 26, 26]           2,416
            Conv2d-3            [-1, 120, 9, 9]          48,120
            Linear-4                   [-1, 84]          10,164
            Linear-5                   [-1, 10]             850
Total params: 61,706
Trainable params: 61,706
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.02
Forward/backward pass size (MB): 0.32
Params size (MB): 0.24
Estimated Total Size (MB): 0.57
----------------------------------------------------------------


In [85]:
writer = SummaryWriter()

count = 0

for ep in range(num_epochs):
    
    model.train()
    running_loss = 0.0
    
    loop = tqdm(train_loader, leave=True)
    for X, y in loop:
        X, y = X.to("cuda"), y.to("cuda")
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        writer.add_scalar("Loss/train", loss, count)
        count += 1
        running_loss += loss.item()
        loss.backward()
        optimizer.step()
        
        loop.set_description(f"Epoch [{ep+1}/{num_epochs}]")
        loop.set_postfix(loss=loss.item())
        
    epoch_loss = running_loss / len(train_loader)
    print(f"\nEpoch {ep+1} 평균 Loss: {epoch_loss:.4f}")

Epoch [1/10]: 100%|██████████| 469/469 [00:30<00:00, 15.56it/s, loss=0.804]



Epoch 1 평균 Loss: 0.8199


Epoch [2/10]: 100%|██████████| 469/469 [00:31<00:00, 14.74it/s, loss=0.802]



Epoch 2 평균 Loss: 0.8157


Epoch [3/10]: 100%|██████████| 469/469 [00:35<00:00, 13.22it/s, loss=0.799]


...


Epoch 8 평균 Loss: 0.8046


Epoch [9/10]: 100%|██████████| 469/469 [00:36<00:00, 12.88it/s, loss=0.817]



Epoch 9 평균 Loss: 0.8037


Epoch [10/10]: 100%|██████████| 469/469 [00:34<00:00, 13.50it/s, loss=0.813]


Epoch 10 평균 Loss: 0.8030


In [ ]:
# due to deprecation of setuptools, you need to downgrade its version <82.0.0 then run, for example:
!uv pip install setuptools==80.10.2
!tensorboard --logdir=runs

Audited 1 package in 8ms


^C


In [88]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for img, y in test_loader:
        img, y = img.to(device), y.to(device)
        
        logits = model(img)
        _, y_pred = torch.max(logits, dim=1)
        
        correct += (y_pred==y).sum().item()
        total += y.size(0)

print(f"test acc: {correct/total}")

test acc: 0.9884
